In [ ]:
from matplotlib import pyplot as plt
import torch
from sklearn.metrics import roc_curve, f1_score, auc
from tools.embs_tools import get_embs, accelerated_cosine_similarity, get_crops_for_id
import numpy as np
import cv2

In [ ]:
all_embs = get_embs("/home/simon/new_outputs/")
print(all_embs.keys())

In [ ]:
accs = 0
fprs = 0
tprs = 0

for path in list(all_embs.keys()):
    thr = 0.466

    avg_fn = lambda embs: np.mean(embs, axis=0)
    embs = torch.stack([torch.tensor(emb) for _, _, emb in all_embs[path]])
    ids = torch.tensor([id for id, _, _ in all_embs[path]])

    embs = embs[:50000]
    ids = ids[:50000]

    similarity = accelerated_cosine_similarity(embs, embs, batch_size=2048)
    similarity_flat = similarity.flatten()
    issame_tracks = (ids.unsqueeze(0) == ids.unsqueeze(1))
    issame_tracks_flat = issame_tracks.flatten()

    def compute_top1_metrics(similarity: torch.Tensor, issame_tracks: torch.Tensor):
        # similarity: NxN tensor where similarity[i,j] is similarity between i-th and j-th samples
        # issame_tracks: NxN binary tensor where issame_tracks[i,j] == 1 if i and j belong to same track

        thr_matches = torch.tensor([(row > 0.466).nonzero()[-1] for row in similarity])
        thr_issame = torch.tensor([issame_tracks[match, idx] for idx, match in enumerate(thr_matches)])

        # Exclude self-matches by setting diagonal to -inf to avoid selecting self as top match
        similarity_no_diag = similarity.clone()
        diag_indices = torch.arange(similarity.size(0))
        similarity_no_diag[diag_indices, diag_indices] = -float('inf')

        # Find top 1 match indices for each row
        top1_indices = torch.argmax(similarity_no_diag, dim=1)
        thr_score = ((thr_matches == top1_indices) | (thr_matches == thr_issame)).sum().item() / len(thr_matches)
        top1_similarities, _ = torch.max(similarity_no_diag, dim=1)

        # Calculate True Positives (TP): top1 identified sample is indeed the same track
        tp = (issame_tracks[diag_indices, top1_indices] == 1).sum().item()

        # Calculate False Positives (FP): top1 identified sample is not the same track
        fp = (issame_tracks[diag_indices, top1_indices] == 0).sum().item()

        # Calculate total positives and negatives excluding diagonal
        total_positive = issame_tracks.sum().item() - similarity.size(0)  # exclude diagonal
        total_negative = issame_tracks.numel() - issame_tracks.sum().item() - similarity.size(0)

        # Compute rates and accuracy
        tpir = tp / total_positive if total_positive > 0 else 0.0
        fpir = fp / total_negative if total_negative > 0 else 0.0
        accuracy = tp / similarity.size(0)

        print(thr_score)
        return tpir, fpir, accuracy

    tpir, fpir, acc = compute_top1_metrics(similarity, issame_tracks)

    accs += acc
    fprs += fpir
    tprs += tpir

print(accs / len(all_embs))
print(tprs / len(all_embs))
print(fprs / len(all_embs))


In [ ]:
path = list(all_embs.keys())[3]
avg_fn = lambda embs: np.mean(embs, axis=0)
embs = torch.stack([torch.tensor(emb) for _, _, emb in all_embs[path]])
ids = torch.tensor([id for id, _, _ in all_embs[path]])

embs = embs[:100000]
ids = ids[:100000]

similarity = accelerated_cosine_similarity(embs, embs, batch_size=2048)
similarity_flat = similarity.flatten()
issame_tracks = (ids.unsqueeze(0) == ids.unsqueeze(1))
issame_tracks_flat = issame_tracks.flatten()

In [ ]:
len(similarity_flat), len(issame_tracks_flat)

In [ ]:
threshold = 0.5
# matches = similarity_flat > threshold
# issame_tracks_flat = issame_tracks_flat > 0
# f1 = f1_score(issame_tracks_flat, matches)
#
# print(f"F1 Score: {f1}")
# print(f"Accuracy: {(matches == issame_tracks_flat).sum() / len(matches)}")

In [ ]:
fprs, tprs, thresholds = roc_curve(issame_tracks_flat, similarity_flat)
print(f"ROC AUC: {auc(fprs, tprs)}")

In [ ]:
issame_tracks_flat = issame_tracks_flat.to("cpu")
similarity_flat = similarity_flat.to("cpu")

same_sims = similarity_flat[issame_tracks_flat].cpu().numpy()
diff_sims = similarity_flat[~issame_tracks_flat].cpu().numpy()

# plot violin plots for same and different tracks
plt.boxplot([same_sims, diff_sims], showmeans=True, showfliers=False)
plt.xticks([1, 2], ['Same Tracks', 'Different Tracks'])
plt.ylabel('Cosine Similarity')
plt.title('Cosine Similarity Distribution for Same and Different Tracks')

In [ ]:
plt.hist(same_sims, bins=50, alpha=0.5, label='Same Tracks', color='blue')#, density=True)
plt.hist(diff_sims, bins=50, alpha=0.5, label='Different Tracks', color='red')#, density=True)
plt.xlabel('Cosine Similarity')
plt.ylabel('Counts')
plt.title('Cosine Similarity Distribution for Same and Different Tracks')
plt.legend()

In [ ]:
similarity.fill_diagonal_(-float('inf'))
top1_sim, top1_idx = similarity.topk(1, dim=1)


print(top1_idx.squeeze(1).shape)

In [ ]:
print(issame_tracks.shape)

In [ ]:
thr = 0.98
top_1 = True

if top_1:
    def compute_top1_metrics(similarity: torch.Tensor, issame_tracks: torch.Tensor):
        # similarity: NxN tensor where similarity[i,j] is similarity between i-th and j-th samples
        # issame_tracks: NxN binary tensor where issame_tracks[i,j] == 1 if i and j belong to same track

        # Exclude self-matches by setting diagonal to -inf to avoid selecting self as top match
        similarity_no_diag = similarity.clone()
        diag_indices = torch.arange(similarity.size(0))
        similarity_no_diag[diag_indices, diag_indices] = -float('inf')

        # Find top 1 match indices for each row
        top1_indices = torch.argmax(similarity_no_diag, dim=1)
        top1_similarities, _ = torch.max(similarity_no_diag, dim=1)

        above_threshold = top1_similarities >= thr

        # Calculate True Positives (TP): top1 identified sample is indeed the same track
        tp = ((issame_tracks[diag_indices, top1_indices] == 1) & above_threshold).sum().item()

        # Calculate False Positives (FP): top1 identified sample is not the same track
        fp = (
                (issame_tracks[diag_indices, top1_indices] == 0) |
                ((issame_tracks[diag_indices, top1_indices] == 1) & (~above_threshold))
        ).sum().item()

        # Calculate total positives and negatives excluding diagonal
        total_positive = issame_tracks.sum().item() - similarity.size(0)  # exclude diagonal
        total_negative = issame_tracks.numel() - issame_tracks.sum().item() - similarity.size(0)

        # Compute rates and accuracy
        tpir = tp / total_positive if total_positive > 0 else 0.0
        fpir = fp / total_negative if total_negative > 0 else 0.0
        accuracy = tp / similarity.size(0)

        return tpir, fpir, accuracy


    tpr, fpr, acc = compute_top1_metrics(similarity, issame_tracks)
else:
    fn = sum(same_sims <= thr)
    fp = sum(diff_sims >= thr)

    fnr = fn / len(similarity_flat)
    fpr = fp / len(similarity_flat)
    # acc = (tp + tn) / len(issame_tracks)
    acc = 0

print(f"True Positive Rate: {tpr}; False Positive Rate: {fpr}")
print(f"Accuracy: {acc}")

In [ ]:
thr = 0.466

fp_ids = torch.ones(similarity.shape, device="cuda", dtype=torch.bool)
fp_ids = fp_ids & ~issame_tracks.to("cuda")
is_same_indices = torch.nonzero(fp_ids, as_tuple=True)
is_same_indices = torch.stack(is_same_indices, dim=1).cpu()
fp_ids = is_same_indices[diff_sims >= thr]

In [ ]:
print("Len of fp_ids: ", fp_ids.shape)
print("Num of indices: ", len(is_same_indices))
len(similarity_flat)

In [ ]:
for a, b in fp_ids[100:102]:
    id_a = ids[a]
    id_b = ids[b]

    track_a_crops = get_crops_for_id(path, id_a)
    track_b_crops = get_crops_for_id(path, id_b)

    print(f"Number of crops for class {id_a}: {len(track_a_crops)}")
    print(f"Number of crops for class {id_b}: {len(track_b_crops)}")
    print("Similarity scores:", similarity[a, b].item())

    # track_a_crops = [track_a_crops[0], track_a_crops[-1]] if len(track_a_crops) > 1 else track_a_crops
    # track_b_crops = [track_b_crops[0], track_b_crops[-1]] if len(track_b_crops) > 1 else track_b_crops

    for crop_a, crop_b in zip(track_a_crops, track_b_crops):
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(cv2.cvtColor(crop_a, cv2.COLOR_BGR2RGB))
        axes[0].set_title('Crop A')
        axes[0].axis('off')

        axes[1].imshow(cv2.cvtColor(crop_b, cv2.COLOR_BGR2RGB))
        axes[1].set_title('Crop B')
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()

In [ ]:
range = 0.9

plt.xlim([range - 1.0, range])
plt.ylim([1.0 - range, 2.0 - range])
plt.plot(fprs, tprs)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.grid(True)
plt.show()

In [ ]:
plt.show()
plt.figure(figsize=(8, 6))
plt.plot(thresholds, tprs, label='True Positive Rate (TPR)')
plt.plot(thresholds, fprs, label='False Positive Rate (FPR)')
plt.plot(thresholds, tprs - fprs, label='TPR - FPR', linestyle='--')
plt.xlim([0.0, 1.0])
plt.xlabel('Threshold')
plt.ylabel('Rate')
plt.title('TPR and FPR vs. Threshold')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
J = tprs - fprs
ix = np.argmax(J)
best_threshold = thresholds[ix]
print(f'Best Threshold: {best_threshold}')